## Dynamic breakpoints

### Goals
Breakpoints are set by the developer on a specific node during graph compilation.

But, sometimes it is helpful to allow the graph dynamically interrupt itself!

This is an internal breakpoint, and can be achieved using NodeInterrupt.

This has a few specific benefits:

(1) You can do it conditionally (from inside a node based on developer-defined logic).

(2) You can communicate to the user why it's interrupted (by passing whatever you want to the NodeInterrupt).

Let's create a graph where a NodeInterrupt is thrown based on the length of the input.

In [ ]:
from IPython.display import Image, display

from typing_extensions import TypedDict
from langgraph.checkpoint.memory import MemorySaver
from langgraph.errors import NodeInterrupt
from langgraph.graph import START, END, StateGraph

class State(TypedDict):
    input: str

def step_1(state: State) -> State:
    print("---Step 1---")
    return state

def step_2(state: State) -> State:
    # Let's optionally raise a NodeInterrupt if the length of the input is longer than 5 characters
    if len(state['input']) > 5:
        raise NodeInterrupt(f"Received input that is longer than 5 characters: {state['input']}")
    
    print("---Step 2---")
    return state

def step_3(state: State) -> State:
    print("---Step 3---")
    return state

builder = StateGraph(State)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)
builder.add_edge(START, "step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

# Set up memory
memory = MemorySaver()

# Compile the graph with memory
graph = builder.compile(checkpointer=memory)

# View
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
initial_input = {"input": "hello world"}
thread_config = {"configurable": {"thread_id": "1"}}

# Run the graph until the first interruption
for event in graph.stream(initial_input, thread_config, stream_mode="values"):
    print(event)

In [ ]:
state = graph.get_state(thread_config)
print(state.next)

In [ ]:

print(state.tasks)

In [ ]:
for event in graph.stream(None, thread_config, stream_mode="values"):
    print(event)

In [ ]:
state = graph.get_state(thread_config)
print(state.next)

In [ ]:
graph.update_state(
    thread_config,
    {"input": "hi"},
)

In [ ]:
for event in graph.stream(None, thread_config, stream_mode="values"):
    print(event)